# Modelo: Árbol de Decisión

**Documentación individual** de uno de los 6 modelos probados en el proyecto Bet-IPS.

Este notebook reutiliza la misma preparación de datos del notebook principal (`Prediccion_WC2026.ipynb`) — carga, limpieza, EDA, ranking FIFA, forma reciente, dataset y split temporal — y entrena **solo Árbol de Decisión** para verlo de forma aislada.

> Pensado para **Google Colab**. Sube los mismos archivos de datos que el notebook principal.

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, bisect

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", palette="deep")
print("Librerías cargadas.")

# AVANCE I — Exploración y limpieza de datos

## 2. Carga de datos

Sube **de una vez** todos los archivos: `results.csv`, `fifa_ranking 1993-2018.csv` y los `rankingFifaXXXX.json` (2019, 2022, 2023, 2026; el 2018 es opcional).

In [ ]:
from google.colab import files
archivos = files.upload()
print("\nArchivos recibidos:", list(archivos.keys()))

In [ ]:
df = pd.read_csv("results.csv", parse_dates=["date"])
df_original = df.copy()
print(f"Partidos cargados: {len(df):,}")
df.head()

## 3. Exploración inicial

In [ ]:
print("Dimensiones:", df.shape)
print("Rango de fechas:", df["date"].min().date(), "->", df["date"].max().date())
df.info()

In [ ]:
# Nulos y duplicados
print("Nulos por columna:")
print(df.isnull().sum())
print("\nFilas duplicadas:", df.duplicated().sum())

## 4. Limpieza de datos

Único problema real en este dataset: **nulos en los marcadores** (partidos sin resultado). Los eliminamos porque no se puede predecir un resultado inexistente. No hay duplicados ni marcadores negativos, así que no hay más que limpiar.

In [ ]:
# Gráfico ANTES: nulos por columna
df.isnull().sum().plot(kind="bar", color="#d9534f", figsize=(9, 4), title="ANTES — Nulos por columna")
plt.tight_layout(); plt.show()

In [ ]:
# Limpieza: quitar filas sin marcador y pasar goles a entero
antes = len(df)
df = df.dropna(subset=["home_score", "away_score"]).reset_index(drop=True)
df["home_score"] = df["home_score"].astype(int)
df["away_score"] = df["away_score"].astype(int)
print(f"Filas eliminadas: {antes - len(df):,}  |  Restantes: {len(df):,}")

In [ ]:
# Gráfico DESPUÉS: nulos por columna
df.isnull().sum().plot(kind="bar", color="#5cb85c", figsize=(9, 4), title="DESPUÉS — Nulos por columna")
plt.tight_layout(); plt.show()

## 5. Gráficos exploratorios

In [ ]:
# Partidos por año
df["anio"] = df["date"].dt.year
df.groupby("anio").size().plot(kind="line", figsize=(10, 4), title="Partidos internacionales por año")
plt.ylabel("Partidos"); plt.tight_layout(); plt.show()

In [ ]:
# Resultado del local en todo el histórico
res = np.where(df["home_score"] > df["away_score"], "Gana local",
               np.where(df["home_score"] == df["away_score"], "Empate", "Gana visitante"))
pd.Series(res).value_counts().plot(kind="bar", figsize=(8, 4), title="Resultado del local (histórico)")
plt.ylabel("Partidos"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

# AVANCE II — Modelos de clasificación (todas las selecciones)

## 6. Enfoque

Entrenamos **un solo modelo con todas las selecciones** (~49 mil partidos). Cada partido se describe desde la perspectiva del **equipo local** y la variable objetivo es:

- **0 = gana local**, **1 = empate**, **2 = gana visitante**.

**Características:** para el local y el visitante calculamos su **forma reciente** (últimos partidos) y su **ranking FIFA** vigente antes del partido, más la **diferencia** entre ambos. La diferencia de nivel entre los dos equipos es la señal más fuerte.

**Validación (split temporal realista):**
- **Entrenamiento:** toda la historia hasta el **fin de la fase de grupos del Mundial 2026** (2026-06-27).
- **Prueba:** los **playoffs del Mundial 2026** (desde 2026-06-28), que el modelo nunca vio.

Ninguna característica usa el marcador del propio partido: todo se calcula con información anterior.

## 7. Ranking FIFA

Combinamos el CSV mensual (1993–2018) con las fotos JSON (fechas posteriores). Para cada equipo y fecha tomamos el ranking vigente **más reciente anterior** al partido.

In [ ]:
# Equivalencias de nombres entre results.csv y las fuentes de ranking
EQUIV = {
    "United States": "USA", "South Korea": "Korea Republic", "North Korea": "Korea DPR",
    "Iran": "IR Iran", "Ivory Coast": "Côte d'Ivoire", "China": "China PR",
    "DR Congo": "Congo DR", "Cape Verde": "Cape Verde Islands",
}

# Fuente 1: CSV mensual 1993-2018 -> {equipo: (fechas, [(rank, puntos)])}
rk_csv = pd.read_csv("fifa_ranking 1993-2018.csv", parse_dates=["rank_date"]).sort_values("rank_date")
hist = {}
for eq, g in rk_csv.groupby("country_full"):
    hist[eq] = (list(g["rank_date"]), list(zip(g["rank"].astype(int), g["total_points"].astype(float))))

# Fuente 2: fotos JSON para fechas posteriores a jun-2018
def cargar_json(anio):
    data = json.load(open(f"rankingFifa{anio}.json", encoding="utf-8"))
    tabla = {}
    for r in data["Results"]:
        if r["Rank"] is not None:
            tabla[r["TeamName"][0]["Description"].strip()] = (r["Rank"], r["TotalPoints"])
    return tabla

anios_json = [2018, 2019, 2022, 2023, 2026]
rk_json = {a: cargar_json(a) for a in anios_json if f"rankingFifa{a}.json" in archivos}

RANK_NA = int(rk_csv["rank"].max()) + 1   # valor neutro para desconocidos
CORTE = pd.Timestamp("2018-06-07")        # última fecha del CSV
print("Ranking listo. CSV hasta", CORTE.date(), "| fotos JSON:", list(rk_json.keys()))

In [ ]:
def ranking(equipo, fecha):
    """Devuelve (rank, puntos) FIFA vigentes antes de 'fecha'. Rank neutro si no hay dato."""
    eq = EQUIV.get(equipo, equipo)
    fecha = pd.Timestamp(fecha)
    if fecha <= CORTE and eq in hist:
        fechas, valores = hist[eq]
        pos = bisect.bisect_right(fechas, fecha) - 1
        if pos >= 0:
            return valores[pos]
    candidatos = [a for a in rk_json if a <= fecha.year]
    if candidatos and eq in rk_json[max(candidatos)]:
        return rk_json[max(candidatos)][eq]
    if eq in hist:
        return hist[eq][1][-1]
    return RANK_NA, 0.0

# Prueba
print("Brazil 2022-11-20:", ranking("Brazil", "2022-11-20"))

## 8. Forma reciente de cada selección

Para medir "cómo llega" cada equipo, calculamos el promedio de **puntos** (3 victoria, 1 empate, 0 derrota) y de **goles a favor** en sus últimos `N` partidos, mirando solo el pasado. Lo hacemos para todas las selecciones a la vez.

In [ ]:
N = 5  # ventana de forma reciente

# Pasamos cada partido a formato "una fila por equipo" para calcular su forma
local = df[["date", "home_team", "home_score", "away_score"]].copy()
local.columns = ["date", "equipo", "gf", "gc"]
visita = df[["date", "away_team", "away_score", "home_score"]].copy()
visita.columns = ["date", "equipo", "gf", "gc"]

largo = pd.concat([local, visita]).sort_values("date").reset_index(drop=True)
largo["pts"] = np.where(largo["gf"] > largo["gc"], 3, np.where(largo["gf"] == largo["gc"], 1, 0))

# Forma = promedio de los N partidos ANTERIORES (shift para no incluir el actual)
g = largo.groupby("equipo")
largo["forma_pts"] = g["pts"].transform(lambda s: s.shift(1).rolling(N, min_periods=1).mean())
largo["forma_gf"]  = g["gf"].transform(lambda s: s.shift(1).rolling(N, min_periods=1).mean())

# Diccionario {(fecha, equipo): (forma_pts, forma_gf)} para consultarlo rápido
forma = {(r.date, r.equipo): (r.forma_pts, r.forma_gf) for r in largo.itertuples()}
print("Forma calculada para", largo['equipo'].nunique(), "selecciones.")

## 9. Construcción del dataset de modelado

Para cada partido armamos las características del **local** y del **visitante** (forma + ranking) y sus **diferencias**, más si el partido fue en campo neutral. El objetivo es el resultado del local (0/1/2).

In [ ]:
d = df.copy()

# Objetivo: 0 gana local, 1 empate, 2 gana visitante
d["objetivo"] = np.where(d["home_score"] > d["away_score"], 0,
                         np.where(d["home_score"] == d["away_score"], 1, 2))

# Ranking de cada equipo (vigente antes del partido)
d["rank_local"]  = [ranking(t, f)[0] for t, f in zip(d["home_team"], d["date"])]
d["rank_visita"] = [ranking(t, f)[0] for t, f in zip(d["away_team"], d["date"])]
d["pts_local"]   = [ranking(t, f)[1] for t, f in zip(d["home_team"], d["date"])]
d["pts_visita"]  = [ranking(t, f)[1] for t, f in zip(d["away_team"], d["date"])]

# Forma de cada equipo (calculada en la sección 8)
d["forma_local"]  = [forma.get((f, t), (np.nan, np.nan))[0] for t, f in zip(d["home_team"], d["date"])]
d["forma_visita"] = [forma.get((f, t), (np.nan, np.nan))[0] for t, f in zip(d["away_team"], d["date"])]

# Diferencias (local - visitante)
d["dif_rank"]  = d["rank_visita"] - d["rank_local"]   # positivo = local mejor ranking
d["dif_pts"]   = d["pts_local"] - d["pts_visita"]
d["dif_forma"] = d["forma_local"] - d["forma_visita"]
d["neutral"]   = d["neutral"].astype(int)

# Quitamos partidos sin forma (primeros de cada equipo)
d = d.dropna(subset=["forma_local", "forma_visita"]).reset_index(drop=True)
print(f"Partidos listos para el modelo: {len(d):,}")
d[["date", "home_team", "away_team", "dif_rank", "dif_forma", "neutral", "objetivo"]].head()

## 10. División temporal: entrenar hasta la fase de grupos, evaluar en playoffs

- **Train:** todo hasta **2026-06-27** (fin de fase de grupos del Mundial 2026).
- **Test:** los **playoffs** del Mundial 2026 (`FIFA World Cup`, desde **2026-06-28**).

In [ ]:
features = ["dif_rank", "dif_pts", "dif_forma", "rank_local", "rank_visita",
            "forma_local", "forma_visita", "neutral"]

FIN_GRUPOS = pd.Timestamp("2026-06-27")

es_playoff = (d["tournament"] == "FIFA World Cup") & (d["date"] >= "2026-06-28")
train = d[d["date"] <= FIN_GRUPOS]
test  = d[es_playoff]

X_train, y_train = train[features], train["objetivo"]
X_test,  y_test  = test[features],  test["objetivo"]

print(f"Entrenamiento: {len(X_train):,} partidos (hasta fase de grupos WC2026)")
print(f"Prueba (playoffs WC2026): {len(X_test)} partidos")
print("\nResultados reales en los playoffs:")
print(y_test.map({0: 'Gana local', 1: 'Empate', 2: 'Gana visitante'}).value_counts())

In [ ]:
# Escalado de las características (mismo para todos los modelos)
from sklearn.preprocessing import StandardScaler
escalador = StandardScaler().fit(X_train)
X_train_s = escalador.transform(X_train)
X_test_s  = escalador.transform(X_test)

## 11. Correlación de las características con el objetivo

In [ ]:
corr = train[features + ["objetivo"]].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlación características vs objetivo")
plt.tight_layout(); plt.show()

## 12. Entrenamiento y evaluación — Árbol de Decisión

Divide los datos con reglas tipo "si/entonces". Se limita la profundidad (`max_depth=5`) para evitar que memorice el entrenamiento (sobreajuste).

Entrenamos el modelo, calculamos sus métricas (Accuracy, Precision, Recall y F1-Score macro) y mostramos su matriz de confusión.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

clases = ["Gana local", "Empate", "Gana visitante"]

from sklearn.tree import DecisionTreeClassifier

modelo = DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42)
modelo.fit(X_train_s, y_train)
pred = modelo.predict(X_test_s)

print("===== Árbol de Decisión =====")
print(f"Accuracy : {accuracy_score(y_test, pred):.3f}")
print(f"Precision: {precision_score(y_test, pred, average='macro', zero_division=0):.3f}  (macro)")
print(f"Recall   : {recall_score(y_test, pred, average='macro', zero_division=0):.3f}  (macro)")
print(f"F1-Score : {f1_score(y_test, pred, average='macro', zero_division=0):.3f}  (macro)\n")
print(classification_report(y_test, pred, target_names=clases, zero_division=0))

## 13. Matriz de confusión

In [ ]:
cm = confusion_matrix(y_test, pred, labels=[0, 1, 2])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=clases, yticklabels=clases)
plt.title("Matriz de confusión — Árbol de Decisión")
plt.xlabel("Predicho"); plt.ylabel("Real")
plt.tight_layout(); plt.show()

## 14. Predicciones vs resultados reales (playoffs del Mundial 2026)

In [ ]:
nombres = {0: "Gana local", 1: "Empate", 2: "Gana visitante"}
comparacion = test[["date", "home_team", "away_team", "home_score", "away_score"]].copy()
comparacion["Real"] = y_test.map(nombres).values
comparacion["Predicho"] = pd.Series(pred, index=comparacion.index).map(nombres)
comparacion["Acierto"] = np.where(comparacion["Real"] == comparacion["Predicho"], "OK", "X")
print(f"Aciertos: {(comparacion['Acierto'] == 'OK').sum()} de {len(comparacion)}")
comparacion

## 15. Conclusión del modelo

Este notebook documenta **Árbol de Decisión** de forma aislada: su entrenamiento, métricas y matriz de confusión sobre los playoffs del Mundial 2026. Para la **comparación de los 6 modelos** y la elección del modelo final, ver el notebook principal `Prediccion_WC2026.ipynb`.

Las métricas se interpretan priorizando el **F1-Score macro** por el desbalance de clases (el empate es la clase minoritaria). La **Accuracy** por sí sola puede ser engañosa.